# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faja27/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** among pages already ranking close to page 1 (positions 4-20) with real search demand behind them, which ones are genuinely underperforming their traffic potential — and does a model that reads content and engagement signals find them better than a rule that only reads impression volume?

**Decision it supports:** a content team's weekly triage — which of these near-page-1 pages to review first for a CTR or content fix, instead of reviewing the biggest pages regardless of whether they actually need fixing.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faja27/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** the anonymized starter CSV, `data/raw/content_refresh_anonymized.csv` (the FlyRank ML Internship dataset sample) — 30,000 content rows, 90-day trailing aggregates (impressions, clicks, engagement), plus static content attributes (word count, competition, content type, intent).

**My lane's slice:** 12,701 rows (42.3%) — `position_tier` in `{page_1, striking}` (positions 4-20) **and**
`impression_tier` in `{moderate, good, excellent}` (real search demand). The other 57.7% (deep pages, or
pages with no meaningful demand) is out of scope for this analysis by design, not by accident.

**Excluded, and why:**
- `ctr`, `clicks_90d` — these define the label; using them as features would be circular.
- `trend_direction`, `trend_pct` — label-derived in the wider dataset conventions; off-limits regardless of lane.
- 30-day comparison windows — they underlie `trend_pct`; excluded to keep a clean line even though they
  aren't the label themselves.
- `provider_used`, `model_used` — dictionary-flagged as not model features.
- `client_id`, `content_id` — identifiers only, used for grouping (never as features).

**Public-safe:** no client names, no raw queries, no URLs anywhere in this work — `client_id`/`content_id`
are already anonymized hashes in the source release.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 140)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

in_range = df["position_tier"].isin(["page_1", "striking"])
has_demand = df["impression_tier"].isin(["moderate", "good", "excellent"])
is_candidate = in_range & has_demand
df["quick_win_score"] = np.where(is_candidate, df["impressions_90d"], 0)
cand = df[is_candidate].copy().reset_index(drop=True)

print(f"total rows: {len(df):,}")
print(f"lane rows:  {len(cand):,} ({len(cand)/len(df):.1%})")
print(f"clients in lane: {cand['client_id'].nunique()} of {df['client_id'].nunique()} total")

total rows: 30,000
lane rows:  12,701 (42.3%)
clients in lane: 29 of 32 total


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `1` if a candidate's own `ctr` is below the weighted CTR benchmark for `page_1`+`striking` combined (`sum(clicks)/sum(impressions)`, not a mean of rates) — an observed, non-circular yes/no target, not a ranking invented after the fact.

**Baseline:** rank candidates by `impressions_90d` alone (the "obvious" quick-win rule — biggest audience in range, fix first). Transparent, one line of logic, the thing any model has to beat.

**Model:** Logistic Regression and Random Forest, both on the same 13 numeric + 7 categorical features (content, competition, engagement, and position/demand tier signals — full list in the repo's Week-5 notebook). Chosen from the toolkit's "yes/no with an observed label" row: readable first, stronger second, letting the comparison decide which earns its complexity.

**Validation design:** `GroupShuffleSplit` by `client_id` (80/20, seed=42) — different clients have different sites and content styles, so a plain random split risks the model partly recognizing a client's house style instead of learning generalizable signal. Verified zero client overlap between train and test.

**Leakage checks:** confirmed no label-source columns (`ctr`, `clicks_90d`) or FlyRank product flags in the feature set; confirmed the harness itself reacts correctly by deliberately injecting `ctr` as a feature and watching AUC jump to 1.0 (full detail in the repo's Week-6 notebook) — the honest 0.70 AUC reported below is a real number, not a blind test.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEED = 42

bench_pool = df[df["position_tier"].isin(["page_1", "striking"])]
benchmark_ctr = 100 * bench_pool["clicks_90d"].sum() / bench_pool["impressions_90d"].sum()
cand["label"] = (cand["ctr"] < benchmark_ctr).astype(int)
cand["has_keyword_data"] = cand["search_volume"].notna().astype(int)

num_feats = ["search_volume", "competition", "cpc", "word_count", "char_count",
             "content_age_days", "days_since_last_update", "impressions_90d",
             "engagement_rate", "scroll_rate", "ai_traffic_pct", "avg_position", "has_keyword_data"]
cat_feats = ["competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
             "position_tier", "impression_tier"]

X = cand[num_feats + cat_feats].copy()
for c in cat_feats:
    X[c] = X[c].fillna("unknown")
y = cand["label"].values
groups = cand["client_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))
overlap = set(cand.iloc[train_idx]["client_id"]) & set(cand.iloc[test_idx]["client_id"])

print(f"benchmark CTR: {benchmark_ctr:.3f}%")
print(f"train: {len(train_idx):,} rows / {cand.iloc[train_idx]['client_id'].nunique()} clients")
print(f"test:  {len(test_idx):,} rows / {cand.iloc[test_idx]['client_id'].nunique()} clients")
print(f"client overlap (must be empty): {overlap}")

forbidden = {"ctr", "clicks_90d", "trend_direction", "trend_pct", "health_score",
             "priority_score", "action_type", "needs_ctr_fix", "is_quick_win"}
print(f"forbidden columns in feature set: {forbidden & set(num_feats+cat_feats) or 'none'}")

benchmark CTR: 0.350%
train: 10,355 rows / 23 clients
test:  2,346 rows / 6 clients
client overlap (must be empty): set()
forbidden columns in feature set: none


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Same candidate pool, same grouped test split, same metric family: **precision@K** (of the top K a ranking puts first, how many are actually underperforming) with the base rate reported alongside — a precision number means nothing without it.

In [4]:
from sklearn.metrics import roc_auc_score

pre_scaled = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
])
pre_plain = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
])

log_reg = Pipeline([("pre", pre_scaled), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])
rand_forest = Pipeline([("pre", pre_plain), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1))])

log_reg.fit(X.iloc[train_idx], y[train_idx])
rand_forest.fit(X.iloc[train_idx], y[train_idx])

lr_proba = log_reg.predict_proba(X.iloc[test_idx])[:, 1]
rf_proba = rand_forest.predict_proba(X.iloc[test_idx])[:, 1]
baseline_test = cand.iloc[test_idx]["quick_win_score"].values
y_test = y[test_idx]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"test base rate: {y_test.mean():.3f}\n")
results_table = pd.DataFrame({
    "K": [50, 100, 200],
    "base_rate": [round(y_test.mean(), 3)] * 3,
    "baseline_volume_rule": [round(precision_at_k(baseline_test, y_test, k), 3) for k in (50, 100, 200)],
    "logistic_regression": [round(precision_at_k(lr_proba, y_test, k), 3) for k in (50, 100, 200)],
    "random_forest": [round(precision_at_k(rf_proba, y_test, k), 3) for k in (50, 100, 200)],
})
print(results_table.to_string(index=False))
print(f"\nROC-AUC: LR={roc_auc_score(y_test, lr_proba):.3f}  RF={roc_auc_score(y_test, rf_proba):.3f}")

test base rate: 0.681

  K  base_rate  baseline_volume_rule  logistic_regression  random_forest
 50      0.681                  0.68                0.900          0.940
100      0.681                  0.68                0.910          0.890
200      0.681                  0.66                0.865          0.915

ROC-AUC: LR=0.626  RF=0.698


## 5. Limitations

*What this work cannot claim.*

- **Cross-sectional, not causal.** This ranks who looks like an opportunity in one snapshot. It does not claim that fixing a flagged page *will* produce a specific traffic lift — that needs an experiment.
- **Narrow lane by design.** Covers 42.3% of the portfolio (near-page-1, real-demand pages). Says nothing about deep pages or zero-demand pages.
- **Validated on a handful of clients.** 23 training / 6 held-out clients. A genuinely new client is untested, not just "less certain" — Week-6's before/after check showed a real (if modest) AUC inflation (0.758 → 0.698) when client grouping is ignored, so client identity does carry some signal the model can lean on if allowed to.
- **Modest discrimination, strong ranking.** ROC-AUC (0.63-0.70) is unremarkable, even though precision@K is strong (0.87-0.94) — the models separate the *extremes* well (exactly what a review queue needs) without being excellent at classifying every candidate. Precision@K should not be read as "the model explains the data well."
- **One dataset, one time window.** No repeated sampling or time-based validation yet — numbers here are from a single split; a different seed or a later data pull could move them somewhat.

In [5]:
# quantify the before/after grouping check referenced above, for the record
from sklearn.model_selection import train_test_split

before_train, before_test = train_test_split(np.arange(len(X)), test_size=0.2, random_state=SEED, stratify=y)
rf_before = Pipeline([("pre", pre_plain), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1))])
rf_before.fit(X.iloc[before_train], y[before_train])
auc_before = roc_auc_score(y[before_test], rf_before.predict_proba(X.iloc[before_test])[:, 1])
auc_after = roc_auc_score(y_test, rf_proba)

print(f"AUC, naive random split (client_id ignored): {auc_before:.3f}")
print(f"AUC, grouped split by client_id (honest):    {auc_after:.3f}")
print(f"inflation from ignoring client grouping:      {auc_before - auc_after:+.3f}")

AUC, naive random split (client_id ignored): 0.758
AUC, grouped split by client_id (honest):    0.698
inflation from ignoring client grouping:      +0.060


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Random forest's predicted probability sorts every candidate into an archetype, ranked within archetype by
`probability × impressions_90d` (likelihood of being a real opportunity × size of the audience that benefits):

| archetype | probability | action | actual hit rate |
|---|---|---|---|
| `quick_win_high_confidence` | ≥ 0.70 | prioritize for a CTR/content fix this cycle | 84.0% |
| `quick_win_borderline` | 0.40-0.70 | queue for human review, not auto-actioned | 53.1% (~coin flip) |
| `already_efficient` | < 0.40 | no action, monitor only | 9.2% |

**Decay/refresh insight:** underperformance climbs with content age within the lane — 41.8% (0-90 days) →
66.9% → 68.7% → **78.3% for 365+ day content** — a concrete case for a refresh cadence, not just a one-time
push.

**No-go list:** never auto-publish or auto-edit from this queue; never treat `already_efficient` as grounds
to deprioritize a page; never act on `borderline` without a human look; never present this ranking as a
client-facing traffic guarantee.

In [6]:
cand["model_proba"] = rand_forest.predict_proba(X)[:, 1]

def archetype(p):
    if p >= 0.70:
        return "quick_win_high_confidence"
    elif p >= 0.40:
        return "quick_win_borderline"
    return "already_efficient"

cand["archetype"] = cand["model_proba"].apply(archetype)
archetype_stats = cand.groupby("archetype").agg(n=("content_id", "size"), hit_rate=("label", "mean"))
print(archetype_stats)

cand["age_bucket"] = pd.cut(cand["content_age_days"], bins=[0, 90, 180, 365, 100000],
                             labels=["0-90d", "91-180d", "181-365d", "365d+"])
decay_table = cand.groupby("age_bucket", observed=True)["label"].mean()
print("\ndecay/refresh insight:")
print(decay_table)

                              n  hit_rate
archetype                                
already_efficient           284  0.091549
quick_win_borderline       5153  0.531147
quick_win_high_confidence  7264  0.840033

decay/refresh insight:
age_bucket
0-90d       0.418367
91-180d     0.669458
181-365d    0.687350
365d+       0.782784
Name: label, dtype: float64


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Three charts, each with one message, saved to `work/figures/` for the deployed paper to embed with relative paths — plus a metrics JSON to `work/outputs/` as the receipt every number above traces back to.

In [7]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/figures", exist_ok=True)
os.makedirs("work/outputs", exist_ok=True)

# chart 1: precision@K, model vs baseline
fig, ax = plt.subplots(figsize=(6.5, 4))
ks = [50, 100, 200]
width = 0.25
x = np.arange(len(ks))
ax.bar(x - width, results_table["baseline_volume_rule"], width, label="Baseline (volume rule)", color="#999999")
ax.bar(x, results_table["logistic_regression"], width, label="Logistic Regression", color="#4C72B0")
ax.bar(x + width, results_table["random_forest"], width, label="Random Forest", color="#55A868")
ax.axhline(y_test.mean(), color="black", linestyle="--", linewidth=1, label=f"base rate ({y_test.mean():.2f})")
ax.set_xticks(x); ax.set_xticklabels([f"K={k}" for k in ks])
ax.set_ylabel("precision@K"); ax.set_ylim(0, 1)
ax.set_title("Model vs. baseline: precision@K on a grouped holdout split")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("work/figures/capstone_precision_comparison.png", dpi=150)
plt.close(fig)

# chart 2: decay/refresh insight
fig, ax = plt.subplots(figsize=(6, 4))
decay_table.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_ylabel("share underperforming benchmark CTR"); ax.set_xlabel("content age")
ax.set_title("Content age vs. underperformance rate\n(decay/refresh insight)")
ax.set_ylim(0, 1); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("work/figures/capstone_decay_refresh.png", dpi=150)
plt.close(fig)

# chart 3: archetype calibration
fig, ax = plt.subplots(figsize=(6, 4))
archetype_stats["hit_rate"].plot(kind="bar", ax=ax, color="#C44E52")
ax.set_ylabel("actual underperformance rate"); ax.set_xlabel("archetype")
ax.set_title("Archetype calibration: predicted vs. actual")
ax.set_ylim(0, 1); plt.xticks(rotation=15, ha="right"); plt.tight_layout()
plt.savefig("work/figures/capstone_archetype_calibration.png", dpi=150)
plt.close(fig)

print("wrote 3 figures to work/figures/")

capstone_metrics = {
    "lane": "volume_quick_win",
    "total_rows": int(len(df)),
    "lane_rows": int(len(cand)),
    "lane_coverage_pct": round(len(cand) / len(df), 3),
    "benchmark_ctr_pct": round(benchmark_ctr, 3),
    "test_base_rate": round(float(y_test.mean()), 3),
    "precision_at_k": results_table.set_index("K").to_dict(orient="index"),
    "auc": {"logistic_regression": round(float(roc_auc_score(y_test, lr_proba)), 3),
            "random_forest": round(float(roc_auc_score(y_test, rf_proba)), 3)},
    "auc_grouping_check": {"naive_random_split": round(float(auc_before), 3),
                            "grouped_split": round(float(auc_after), 3)},
    "archetype_hit_rates": archetype_stats["hit_rate"].round(3).to_dict(),
    "archetype_sizes": archetype_stats["n"].to_dict(),
    "decay_refresh_by_age": decay_table.round(3).to_dict(),
    "clients": {"total": int(cand["client_id"].nunique()),
                "train": int(cand.iloc[train_idx]["client_id"].nunique()),
                "test": int(cand.iloc[test_idx]["client_id"].nunique())},
    "seed": SEED,
}
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(capstone_metrics, f, indent=2)
print("wrote work/outputs/capstone_metrics.json")

ranked_queue = (cand.assign(action_priority_score=cand["model_proba"] * cand["impressions_90d"])
                .sort_values("action_priority_score", ascending=False)
                [["content_id", "client_id", "archetype", "model_proba", "impressions_90d",
                  "action_priority_score", "position_tier", "avg_position", "ctr", "content_age_days"]]
                .reset_index(drop=True))
ranked_queue.insert(0, "rank", ranked_queue.index + 1)
ranked_queue.to_csv("work/outputs/capstone_ranked_queue.csv", index=False)
print(f"wrote {len(ranked_queue):,} rows to work/outputs/capstone_ranked_queue.csv (not committed, by design)")

wrote 3 figures to work/figures/
wrote work/outputs/capstone_metrics.json
wrote 12,701 rows to work/outputs/capstone_ranked_queue.csv (not committed, by design)


## ML-12 — Demo, social post, and employer summary

*The smallest card, done here in the closing cells so it never gets forgotten.*

### 5-minute demo outline

- **Question (0:00-1:00):** FlyRank's own `is_quick_win` flag prioritizes near-page-1 pages by traffic
  volume. Does volume alone actually find pages that are genuinely underperforming — or is it close to a
  coin flip?
- **Method (1:00-2:00):** 30,000 real content rows, lane = near-page-1 + real demand (42% of the portfolio).
  Label = a page's own CTR below its position tier's benchmark. Baseline = rank by volume. Model = logistic
  regression / random forest on content + engagement signals, validated on a client-grouped holdout split.
- **One chart (2:00-3:00):** the precision@K comparison — baseline hugs the 0.68 base rate at every K; both
  models reach 0.87-0.94. *(show `capstone_precision_comparison.png`)*
- **One honest result (3:00-4:00):** the lift is real, but ROC-AUC (0.63-0.70) is modest — the models are
  good at separating the extremes, not at cleanly classifying every page. And a naive random split would
  have overstated it (AUC 0.758 vs. the honest 0.698) — say this out loud, don't wait for a question.
- **One recommendation (4:00-5:00):** ship the archetype-based review queue, not an automated fix — start
  with `quick_win_high_confidence` (84% hit rate), and revisit the refresh cadence for content over a year
  old (78% underperformance rate there).

### Social-post cut
> FlyRank's product flags SEO "quick wins" mostly by traffic volume. Turns out that's barely better than a
> coin flip at finding pages that actually need fixing. In this sample, a classifier trained on content and
> engagement signals — validated with a client-grouped holdout split — pushed precision@50 from ~0.68 to
> ~0.94. Full write-up + reproducible notebooks: [link]. #MachineLearning #SEO

### Employer-facing summary
I built a validated ranking model that finds underperforming near-page-1 content, on a 30,000-row sample of
real production SEO data. Using a client-grouped holdout split and a deliberately injected leakage test to
verify the evaluation itself, the model lifted precision@50 from a 0.68 volume-only baseline to 0.94. The
output ships as a reviewed action playbook with explicit no-go rules, not an automated pipeline.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.